In [ ]:
!pip install seaborn

# The main function of this project is to create an Alternative product recommender system that is able to recommend similar products when the originally intended Item of purchase Is out of stock. There are so many empty fields in the ingredients fields, there may be products that are poorly matched compared to others. In the lines of code below each of them serve a specific purpose in this process. The main columns that are relevant to our usecase for the alternative product recommender system are the following columns, inv_name, pi1_descritpion, pi2_description, dpt_name and than all of the Ingredients, columns which were separated in excel for better product matching. 

# Below we are calling some libraries that will be used for data manipulationa and preprocessing

In [1]:
import pandas as pd
import numpy as np

import nltk
import re
from bs4 import BeautifulSoup
from nltk.stem import PorterStemmer
from nltk.stem.wordnet import WordNetLemmatizer

import seaborn as sns
import matplotlib.pyplot as pltx




In [2]:
df = pd.read_csv("Demo_data.csv", encoding='utf-8')
#pd.set_option("display.max_columns", None)
#pd.set_option("display.max_rows", None)
df


,inv_pk,inv_type,inv_scancode,inv_name,inv_size,inv_receiptalias,inv_discontinued,pi1_description,pi2_description,brd_name,...,Ingredient_106,Ingredient_107,Ingredient_108,Ingredient_109,Ingredient_110,Ingredient_111,Ingredient_112,Ingredient_113,Ingredient_114,Ingredient_115
0,37312,Stock Inventory,11172,Cook Eat Feel Healthy America,1 ct,Cook Eat Feel Healthy America,1,OTHER (MISCELLANEOUS BASE-CODE,Books,Cooking with Phil,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,43576,Stock Inventory,12511530129,Candy Delish Fish OG,2 oz,WS Candy Delish Fish OG 2OZ,1,SHELF STABLE CANDY,SS CANDY NON CHOCOLATE,Wholesome Sweeteners,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,43746,Stock Inventory,23923300347,Winter Squash OG,4 oz,EB Winter Squash OG 4OZ,0,SHELF STABLE BABY FOOD,SS BABY FOOD,Earth's Best,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,43747,Stock Inventory,23923300316,Sweet Potatoes OG,4 oz,"BABY FD,OG2,SWT POT,ST 1",1,SHELF STABLE BABY FOOD,SS BABY FOOD,Earth's Best,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,43748,Stock Inventory,23923300378,Spinach & Potatoes OG,4 oz,Spinach & Potatoes OG,1,SHELF STABLE BABY FOOD,SS BABY FOOD,Earth's Best,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
264,44549,Stock Inventory,73360233418,Seltzer Lime 12pk,12 / 12 fl oz,LAC Seltzer Lime 12PK,0,SHELF STABLE WATER,SS WATER SPARKLING FLAVORED,LaCroix,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
265,44548,Stock Inventory,12993221256,Seltzer Lime 8pk,8 / 12 fl oz,LAC Seltzer Lime 8PK,1,SHELF STABLE WATER,SS WATER SPARKLING FLAVORED,LaCroix,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
266,44555,Stock Inventory,73360772054,Sparkling Water KiwiSandia 8pk,8 / 12 fl oz,SpkgWtr KwSnd 8pk,1,SHELF STABLE WATER,SS WATER SPARKLING FLAVORED,LaCroix,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
267,44554,Stock Inventory,12993102012,Seltzer Grapefruit 12pk,12 / 12 fl oz,LAC SpkgWtr Grpfrt 12PK,0,SHELF STABLE WATER,SS WATER SPARKLING FLAVORED,LaCroix,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
#Here we want to drop all rows in the inv_discontinued column that indicates whether a prodcut is discontued or not, 1 being yes and 0 being no,
#so below we will get all items that are not discontinued which will be relevant for our goal
df_A2 = df[df['inv_discontinued'] != 1]
df_A2

,inv_pk,inv_type,inv_scancode,inv_name,inv_size,inv_receiptalias,inv_discontinued,pi1_description,pi2_description,brd_name,...,Ingredient_106,Ingredient_107,Ingredient_108,Ingredient_109,Ingredient_110,Ingredient_111,Ingredient_112,Ingredient_113,Ingredient_114,Ingredient_115
2,43746,Stock Inventory,23923300347,Winter Squash OG,4 oz,EB Winter Squash OG 4OZ,0,SHELF STABLE BABY FOOD,SS BABY FOOD,Earth's Best,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,43750,Stock Inventory,23923330344,Pouch Squash & Sweet Pea,3 5 oz,EB Pouch Squash & Swt Pea 3 5OZ,0,SHELF STABLE BABY FOOD,SS BABY FOOD,Earth's Best,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,43762,Stock Inventory,23923100602,Infant Formula Sens Iron OG,21 oz,EB Infant Form Sens Iron OG 21OZ,0,SHELF STABLE BABY FOOD,SS BABY FORMULA,Earth's Best,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,43763,Stock Inventory,23923100442,Infant Formula DHA & ARA OG,21 oz,EB Infant Form DHA & ARA 21OZ,0,SHELF STABLE BABY FOOD,SS BABY FORMULA,Earth's Best,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,43803,Stock Inventory,89991516615,Black Forest Uncured Ham,7 oz,PLA Black Forest Uncured Ham 7OZ,0,FROZEN & REFRIGERATED MEAT POU,FZ & RF DELI MEAT,Plainville,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
243,44784,Stock Inventory,42563603755,Honey Raw Wildflower OG,16 oz,FD Honey Raw Wildflowr 16OZ,0,SHELF STABLE SWEETENERS,SS HONEY,Field Day,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
251,44779,Stock Inventory,42563603595,Canola Oil,32 fl oz,FD Canola Oil 32FZ,0,SHELF STABLE OILS & VINEGARS,SS CULINARY OIL CANOLA,Field Day,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
258,44547,Stock Inventory,12993102029,Seltzer Coconut 12pk,12 / 12 fl oz,LAC Seltzer Ccnut 12PK,0,SHELF STABLE WATER,SS WATER SPARKLING FLAVORED,LaCroix,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
264,44549,Stock Inventory,73360233418,Seltzer Lime 12pk,12 / 12 fl oz,LAC Seltzer Lime 12PK,0,SHELF STABLE WATER,SS WATER SPARKLING FLAVORED,LaCroix,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# We are replacing all NaNs under all ingredients to None
df_A3 = df_A2.replace({np.nan: None})
df_A3

,inv_pk,inv_type,inv_scancode,inv_name,inv_size,inv_receiptalias,inv_discontinued,pi1_description,pi2_description,brd_name,...,Ingredient_106,Ingredient_107,Ingredient_108,Ingredient_109,Ingredient_110,Ingredient_111,Ingredient_112,Ingredient_113,Ingredient_114,Ingredient_115
2,43746,Stock Inventory,23923300347,Winter Squash OG,4 oz,EB Winter Squash OG 4OZ,0,SHELF STABLE BABY FOOD,SS BABY FOOD,Earth's Best,...,None,None,None,None,None,None,None,None,None,None
6,43750,Stock Inventory,23923330344,Pouch Squash & Sweet Pea,3 5 oz,EB Pouch Squash & Swt Pea 3 5OZ,0,SHELF STABLE BABY FOOD,SS BABY FOOD,Earth's Best,...,None,None,None,None,None,None,None,None,None,None
15,43762,Stock Inventory,23923100602,Infant Formula Sens Iron OG,21 oz,EB Infant Form Sens Iron OG 21OZ,0,SHELF STABLE BABY FOOD,SS BABY FORMULA,Earth's Best,...,None,None,None,None,None,None,None,None,None,None
16,43763,Stock Inventory,23923100442,Infant Formula DHA & ARA OG,21 oz,EB Infant Form DHA & ARA 21OZ,0,SHELF STABLE BABY FOOD,SS BABY FORMULA,Earth's Best,...,None,None,None,None,None,None,None,None,None,None
21,43803,Stock Inventory,89991516615,Black Forest Uncured Ham,7 oz,PLA Black Forest Uncured Ham 7OZ,0,FROZEN & REFRIGERATED MEAT POU,FZ & RF DELI MEAT,Plainville,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
243,44784,Stock Inventory,42563603755,Honey Raw Wildflower OG,16 oz,FD Honey Raw Wildflowr 16OZ,0,SHELF STABLE SWEETENERS,SS HONEY,Field Day,...,None,None,None,None,None,None,None,None,None,None
251,44779,Stock Inventory,42563603595,Canola Oil,32 fl oz,FD Canola Oil 32FZ,0,SHELF STABLE OILS & VINEGARS,SS CULINARY OIL CANOLA,Field Day,...,None,None,None,None,None,None,None,None,None,None
258,44547,Stock Inventory,12993102029,Seltzer Coconut 12pk,12 / 12 fl oz,LAC Seltzer Ccnut 12PK,0,SHELF STABLE WATER,SS WATER SPARKLING FLAVORED,LaCroix,...,None,None,None,None,None,None,None,None,None,None
264,44549,Stock Inventory,73360233418,Seltzer Lime 12pk,12 / 12 fl oz,LAC Seltzer Lime 12PK,0,SHELF STABLE WATER,SS WATER SPARKLING FLAVORED,LaCroix,...,None,None,None,None,None,None,None,None,None,None


In [5]:
#Here we are dropping columns that are irrelevant in the time being
df_A4 = df_A3.drop(columns = ["inv_type", "inv_receiptalias", "inv_lastreceived", "del_ingredientList", "inv_size", "inv_discontinued", "inv_lastsold", "sib_baseprice"])
df_A4

,inv_pk,inv_scancode,inv_name,pi1_description,pi2_description,brd_name,dpt_name,Ingredient_1,Ingredient_2,Ingredient_3,...,Ingredient_106,Ingredient_107,Ingredient_108,Ingredient_109,Ingredient_110,Ingredient_111,Ingredient_112,Ingredient_113,Ingredient_114,Ingredient_115
2,43746,23923300347,Winter Squash OG,SHELF STABLE BABY FOOD,SS BABY FOOD,Earth's Best,Packaged Grocery,ORGANIC WINTER SQUASH,None,None,...,None,None,None,None,None,None,None,None,None,None
6,43750,23923330344,Pouch Squash & Sweet Pea,SHELF STABLE BABY FOOD,SS BABY FOOD,Earth's Best,Packaged Grocery,ORGANIC BUTTERNUT SQUASH PUREE,None,None,...,None,None,None,None,None,None,None,None,None,None
15,43762,23923100602,Infant Formula Sens Iron OG,SHELF STABLE BABY FOOD,SS BABY FORMULA,Earth's Best,Packaged Grocery,ORGANIC GLUCOSE SYRUP SOLIDS,ORGANIC PALM OIL OR PALM OLEIN,ORGANIC WHEY PROTEIN CONCENTRATE,...,None,None,None,None,None,None,None,None,None,None
16,43763,23923100442,Infant Formula DHA & ARA OG,SHELF STABLE BABY FOOD,SS BABY FORMULA,Earth's Best,Packaged Grocery,ORGANIC LACTOSE,ORGANIC NONFAT MILK,ORGANIC OILS (ORGANIC PALM OR PALM OLEIN,...,None,None,None,None,None,None,None,None,None,None
21,43803,89991516615,Black Forest Uncured Ham,FROZEN & REFRIGERATED MEAT POU,FZ & RF DELI MEAT,Plainville,Meat,FRESH HAM,WATER,EVAPORATED CANE SYRUP,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
243,44784,42563603755,Honey Raw Wildflower OG,SHELF STABLE SWEETENERS,SS HONEY,Field Day,Packaged Grocery,100% ORGANIC GRADE A HONEY,None,None,...,None,None,None,None,None,None,None,None,None,None
251,44779,42563603595,Canola Oil,SHELF STABLE OILS & VINEGARS,SS CULINARY OIL CANOLA,Field Day,Packaged Grocery,100% EXPELLER (MECHANICALLY) PRESSED REFINED N...,None,None,...,None,None,None,None,None,None,None,None,None,None
258,44547,12993102029,Seltzer Coconut 12pk,SHELF STABLE WATER,SS WATER SPARKLING FLAVORED,LaCroix,Packaged Grocery,ONLY CARBONATED WATER,NATURALLY ESSENCED*,None,...,None,None,None,None,None,None,None,None,None,None
264,44549,73360233418,Seltzer Lime 12pk,SHELF STABLE WATER,SS WATER SPARKLING FLAVORED,LaCroix,Packaged Grocery,ONLY CARBONATED WATER,NATURALLY ESSENCED*,None,...,None,None,None,None,None,None,None,None,None,None


In [6]:
df_A1 = df_A4.replace({np.nan: None})
df_A1

,inv_pk,inv_scancode,inv_name,pi1_description,pi2_description,brd_name,dpt_name,Ingredient_1,Ingredient_2,Ingredient_3,...,Ingredient_106,Ingredient_107,Ingredient_108,Ingredient_109,Ingredient_110,Ingredient_111,Ingredient_112,Ingredient_113,Ingredient_114,Ingredient_115
2,43746,23923300347,Winter Squash OG,SHELF STABLE BABY FOOD,SS BABY FOOD,Earth's Best,Packaged Grocery,ORGANIC WINTER SQUASH,None,None,...,None,None,None,None,None,None,None,None,None,None
6,43750,23923330344,Pouch Squash & Sweet Pea,SHELF STABLE BABY FOOD,SS BABY FOOD,Earth's Best,Packaged Grocery,ORGANIC BUTTERNUT SQUASH PUREE,None,None,...,None,None,None,None,None,None,None,None,None,None
15,43762,23923100602,Infant Formula Sens Iron OG,SHELF STABLE BABY FOOD,SS BABY FORMULA,Earth's Best,Packaged Grocery,ORGANIC GLUCOSE SYRUP SOLIDS,ORGANIC PALM OIL OR PALM OLEIN,ORGANIC WHEY PROTEIN CONCENTRATE,...,None,None,None,None,None,None,None,None,None,None
16,43763,23923100442,Infant Formula DHA & ARA OG,SHELF STABLE BABY FOOD,SS BABY FORMULA,Earth's Best,Packaged Grocery,ORGANIC LACTOSE,ORGANIC NONFAT MILK,ORGANIC OILS (ORGANIC PALM OR PALM OLEIN,...,None,None,None,None,None,None,None,None,None,None
21,43803,89991516615,Black Forest Uncured Ham,FROZEN & REFRIGERATED MEAT POU,FZ & RF DELI MEAT,Plainville,Meat,FRESH HAM,WATER,EVAPORATED CANE SYRUP,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
243,44784,42563603755,Honey Raw Wildflower OG,SHELF STABLE SWEETENERS,SS HONEY,Field Day,Packaged Grocery,100% ORGANIC GRADE A HONEY,None,None,...,None,None,None,None,None,None,None,None,None,None
251,44779,42563603595,Canola Oil,SHELF STABLE OILS & VINEGARS,SS CULINARY OIL CANOLA,Field Day,Packaged Grocery,100% EXPELLER (MECHANICALLY) PRESSED REFINED N...,None,None,...,None,None,None,None,None,None,None,None,None,None
258,44547,12993102029,Seltzer Coconut 12pk,SHELF STABLE WATER,SS WATER SPARKLING FLAVORED,LaCroix,Packaged Grocery,ONLY CARBONATED WATER,NATURALLY ESSENCED*,None,...,None,None,None,None,None,None,None,None,None,None
264,44549,73360233418,Seltzer Lime 12pk,SHELF STABLE WATER,SS WATER SPARKLING FLAVORED,LaCroix,Packaged Grocery,ONLY CARBONATED WATER,NATURALLY ESSENCED*,None,...,None,None,None,None,None,None,None,None,None,None


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
# Here we dynamically combine all text columns into a single text field
text_columns = [col for col in df_A1.columns if col not in ['inv_scancode', 'inv_pk', 'brd_name']]  # Exclude certain columns
df_A1['Combined_Text'] = df_A1[text_columns].apply(lambda row: ' '.join(row.values.astype(str)), axis=1)


# Cosine similarity measures the similarity between two non-zero vectors by calculating the cosine of the angle between them.
# It is often used in text analysis to measure document similarity. The value of cosine similarity ranges from -1 to 1,
# where 1 indicates identical vectors, 0 indicates orthogonality (no similarity), and -1 indicates diametrically opposed vectors

In [9]:
from sklearn.feature_extraction.text import CountVectorizer
corpus = df_A1['Combined_Text']
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)
X

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 1437 stored elements and shape (61, 487)>

In [10]:
df_A1 = df_A1.reset_index(drop=True)  # Ensure alignment
tfidf_matrix = vectorizer.fit_transform(df_A1['Combined_Text'])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
cosine_sim

array([[1.        , 0.99961595, 0.96323889, ..., 0.99860512, 0.99860512,
        0.99860512],
       [0.99961595, 1.        , 0.96310723, ..., 0.99860512, 0.99860512,
        0.99860512],
       [0.96323889, 0.96310723, 1.        , ..., 0.96138752, 0.96125469,
        0.96125469],
       ...,
       [0.99860512, 0.99860512, 0.96138752, ..., 1.        , 0.99992183,
        0.99992183],
       [0.99860512, 0.99860512, 0.96125469, ..., 0.99992183, 1.        ,
        0.99992183],
       [0.99860512, 0.99860512, 0.96125469, ..., 0.99992183, 0.99992183,
        1.        ]], shape=(61, 61))

In [11]:
#Here we display it as a dataframe to showcase which products closely align with one another
similarity_A1 = pd.DataFrame(cosine_sim, index=df_A1['inv_name'], columns=df_A1['inv_name'])
pd.set_option("display.max_columns", None)
similarity_A1

inv_name,Winter Squash OG,Pouch Squash & Sweet Pea,Infant Formula Sens Iron OG,Infant Formula DHA & ARA OG,Black Forest Uncured Ham,Organic Avocados,Organic Navel Oranges,Nasal Cleansing Pot,Organic Decaf Breakfast Blend Coffee,Gum Pouch Spearmint,Gum Pouch Wintergreen,Smooth Shine Apple Conditioner,Potato Chips Original,Potato Chips Original,Sweet Drops Caramel,Sweet Drops Vanilla,Mexican Fiesta,Fudge Mint Cookies,Fudge Striped Cookies,Chocolate Chip Cookies,Organic Figgy Pops Choco Crunch,Concentrate Cranberry OG,Spray Disinfectant Lavander Vanilla,Spray Disinfectant Fresh Citrus,Spray Disinfectant Eucalyptus,Matzo Whole Wheat Passover,Seltzer Cranberry Clementine,Potato Chips Sour Cream Onion,Organic Dark Chocolate Bar,Milk Chocolate Pretzel Bar,Crispy Onion,Organic 85% Pure Dark Chocolate Bar,Crispbread Chesnut OG,Real Butter Popcorn,Condensed Sweetened Coconut Milk,Sauce Indian Goan Coconut,Mixed Berry Protein Bar,Mint Chocolate Protein Bar,Blueberry Protein Bar,Multigrain Flax Flatbread,The Double Chocolate Complete Cookie,Pizza Sauce OG GF,Yerba Mate Spark Grapft Gngr,Yerba Mate Sparkling Cran Pom,Yerba Mate Bluephoria,Clsc Gld Yerba Mate Sprklng OG,Crackers Original Sea Salt,Almond Butter Dark Chocolate Cups,Dark Chocolate Crispy Quinoa Gems,Dark Chocolate Peanuts Gems,Real Stick Turkey Pepperoni,Sparkling Water RubyRed Grpfrt,Seltzer Georgia Peach,Seltzer Blueberry Limonade,Honey Raw Wildflower OG,Applesauce Cinnamon OG,Honey Raw Wildflower OG,Canola Oil,Seltzer Coconut 12pk,Seltzer Lime 12pk,Seltzer Grapefruit 12pk
inv_name,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Winter Squash OG,1.000000,0.999616,0.963239,0.963820,0.997408,0.998816,0.998816,0.998463,0.998747,0.999040,0.998343,0.998225,0.998426,0.998426,0.998278,0.998072,0.998809,0.995508,0.996486,0.994698,0.997595,0.998848,0.997774,0.997774,0.998740,0.998566,0.998969,0.998349,0.998721,0.997882,0.998772,0.998416,0.999040,0.998418,0.998565,0.995926,0.998349,0.998065,0.998525,0.989079,0.992526,0.998408,0.998656,0.998579,0.998771,0.998694,0.998617,0.995299,0.996617,0.996430,0.996511,0.998705,0.998969,0.998732,0.998848,0.999078,0.998848,0.998234,0.998605,0.998605,0.998605
Pouch Squash & Sweet Pea,0.999616,1.000000,0.963107,0.963686,0.997408,0.998816,0.998816,0.998463,0.998747,0.999116,0.998423,0.998225,0.998426,0.998426,0.998357,0.998152,0.998809,0.995508,0.996486,0.994790,0.997595,0.998771,0.997774,0.997774,0.998740,0.998566,0.998969,0.998349,0.998721,0.997882,0.998772,0.998416,0.998963,0.998418,0.998565,0.995926,0.998349,0.998065,0.998525,0.989079,0.992724,0.998331,0.998656,0.998579,0.998771,0.998617,0.998617,0.995299,0.996617,0.996430,0.996511,0.998705,0.998969,0.998732,0.998771,0.999001,0.998771,0.998234,0.998605,0.998605,0.998605
Infant Formula Sens Iron OG,0.963239,0.963107,1.000000,0.996346,0.960644,0.963109,0.963109,0.960827,0.962062,0.961670,0.961013,0.960598,0.961079,0.961079,0.960948,0.960750,0.961449,0.962456,0.962476,0.958983,0.965262,0.963313,0.961069,0.961069,0.961208,0.961217,0.961600,0.961006,0.963342,0.961395,0.962515,0.962083,0.962708,0.962026,0.963723,0.963073,0.961445,0.961037,0.961458,0.955677,0.959214,0.963099,0.961301,0.961227,0.961412,0.961396,0.961264,0.965461,0.962723,0.962369,0.960157,0.961346,0.961600,0.961375,0.962523,0.962745,0.962523,0.963264,0.961388,0.961255,0.961255
Infant Formula DHA & ARA OG,0.963820,0.963686,0.996346,1.000000,0.960921,0.963813,0.963813,0.961232,0.962616,0.962086,0.961428,0.961003,0.961495,0.961495,0.961363,0.961165,0.961864,0.961584,0.962049,0.959261,0.966444,0.963626,0.961089,0.961089,0.961617,0.961632,0.962015,0.961421,0.964062,0.961824,0.962532,0.962653,0.963276,0.962044,0.964451,0.963246,0.961726,0.961314,0.961740,0.955260,0.958485,0.963820,0.961716,0.961642,0.961827,0.961813,0.961679,0.966301,0.963489,0.962544,0.960585,0.961761,0.962015,0.961790,0.963091,0.963313,0.963091,0.962648,0.961805,0.961670,0.961670
Black Forest Uncured Ham,0.997408,0.997408,0.960644,0.960921,1.000000,0.997837,0.9978

# Below we define multiple functions, that will serve the purpose of retrieving alternative products, I have 2 different models that I am loolkng at,
# The first as you see eblow is the regulat TFIDF vectorizer using Cosine similarity, 2nd model will be the a Bert model utilziing Cosine Similarity 
# and the Final model I wanted to test out FAISS using Bert model embedding, which is an effecient vector search library designed to quickley find the
# Most similar vecotrs in high-dimenstional spaces. So Far the FAISS model is performating the best so I am still working on it improving its accuracy,
# I may have to tweek its model so it alignes well with the dataset I am using or viceversa.

# Below you can observe the outputs of the models and it will give you the alternative products recommended based on similariy matching. It's not the 
# Best but based on the dataset I am working with, its not bad at all! The end Goal is to be able to run the defined fucntions in the manner where i am
# Able to retrieve all the different products and there recommended alternatives in an excel sheet that I am able to store. 

# This model is using TFIDF Vectorizer and Cosine Similarity

In [ ]:
def get_alternative(product_name, df_A1, similarity_matrix, threshold=0.90):
    # Normalize product name to match DataFram
    product_name = product_name.strip().lower()
    df_A1["inv_name"] = df_A1["inv_name"].str.strip().str.lower()

    # Ensure index consistency
    df_A1 = df_A1.reset_index(drop=True)

    if product_name not in df_A1["inv_name"].values:
        return "Product not found"

    # Get index safely
    idx = df_A1[df_A1["inv_name"] == product_name].index[0]

    if idx >= similarity_matrix.shape[0]:
        return "Error: Index out of bounds. Ensure matrix aligns with df_A1."
    


    # Compute similarity
    similar_products = list(enumerate(similarity_matrix[idx]))

    # Apply threshold and filter out identical products
    filtered_products = [
        (i[0], i[1]) for i in similar_products
        if i[1] >= threshold and df_A1.iloc[i[0]]["inv_name"] != product_name
    ]

    # Sort by similarity score
    filtered_products = sorted(filtered_products, key=lambda x: x[1], reverse=True)

    # Extract top alternatives
    alternatives = [df_A1.iloc[i[0]]["inv_name"] for i in filtered_products]

    return alternatives[:7] if alternatives else "No similar products found"


# Test function
print(get_alternative('', df_A1, cosine_sim))

In [ ]:
!pip install openpyxl

## Official model that will create the exported sheet for review using tfidf vectorizer

In [ ]:
import pandas as pd

def get_alternative(df_A1, similarity_matrix, threshold=0.90, top_n=7):
    # Normalize product names to match DataFrame format
    df_A1["inv_name"] = df_A1["inv_name"].str.strip().str.lower()

    # Initialize an empty list to store all results
    all_results = []

    # Iterate over each product in the dataframe
    for idx, product_name in df_A1.iterrows():
        product_name = product_name["inv_name"]
        
        # Compute similarity for the current product
        similar_products = list(enumerate(similarity_matrix[idx]))

        # Apply threshold and filter out identical products
        filtered_products = [
            (i[0], i[1]) for i in similar_products
            if i[1] >= threshold and df_A1.iloc[i[0]]["inv_name"] != product_name
        ]

        # Sort by similarity score (descending)
        filtered_products = sorted(filtered_products, key=lambda x: x[1], reverse=True)

        # Extract the top_n alternatives
        for i in filtered_products[:top_n]:
            similar_idx = i[0]
            similarity_score = i[1]

            # Get the required data from both the original and similar products
            original_product = df_A1.iloc[idx]["inv_name"]
            similar_product = df_A1.iloc[similar_idx]["inv_name"]
            inv_pk = df_A1.iloc[idx]["inv_pk"]
            inv_scancode = df_A1.iloc[idx]["inv_scancode"]
            similar_inv_scancode = df_A1.iloc[similar_idx]["inv_scancode"]
            similar_inv_pk = df_A1.iloc[similar_idx]["inv_pk"]
            similar_brd_name = df_A1.iloc[similar_idx]["brd_name"]
            similar_dpt_name = df_A1.iloc[similar_idx]["dpt_name"]

            # Store the result as a list
            all_results.append([
                inv_pk, inv_scancode, original_product, similar_product, 
                similarity_score, similar_inv_pk, similar_inv_scancode, similar_brd_name, similar_dpt_name
            ])

    # Convert the list of results into a DataFrame
    results_df = pd.DataFrame(
        all_results, 
        columns=["inv_pk", "inv_scancode", "original_product", "similar_product", 
                 "similarity_score", "similar_inv_pk", "similar_inv_scancode", "similar_brd_name", "similar_dpt_name"]
    )

    # Save the results into an Excel file
    results_df.to_csv("testingCurrent_product_similarity_results.csv", index=False)

    return results_df

# Example usage:
#Assuming you already have df_A1 and cosine_sim matrix
result_df = get_alternative(df_A1, cosine_sim, threshold=0.90, top_n=7)
result_df


In [ ]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors

def get_alternative_fast(df_A1, tfidf_matrix, top_n=7):
    # Normalize product names
    df_A1["inv_name"] = df_A1["inv_name"].str.strip().str.lower()

    # Fit Nearest Neighbors model with cosine distance
    nn_model = NearestNeighbors(n_neighbors=top_n + 1, metric='cosine', algorithm='brute')
    nn_model.fit(tfidf_matrix)

    # Get nearest neighbors for all rows
    distances, indices = nn_model.kneighbors(tfidf_matrix)

    all_results = []

    for idx, neighbors in enumerate(indices):
        original_product = df_A1.iloc[idx]
        for i, neighbor_idx in enumerate(neighbors[1:]):  # Skip the first one (itself)
            similarity_score = 1 - distances[idx][i + 1]  # Convert cosine distance to similarity

            similar_product = df_A1.iloc[neighbor_idx]

            all_results.append([
                original_product["inv_pk"],
                original_product["inv_scancode"],
                original_product["inv_name"],
                similar_product["inv_name"],
                similarity_score,
                similar_product["inv_pk"],
                similar_product["inv_scancode"],
                similar_product["brd_name"],
                similar_product["dpt_name"]
            ])

    results_df = pd.DataFrame(all_results, columns=[
        "inv_pk", "inv_scancode", "original_product", "similar_product",
        "similarity_score", "similar_inv_pk", "similar_inv_scancode",
        "similar_brd_name", "similar_dpt_name"
    ])

    results_df.to_csv("1fast_product_similarity_results.csv", index=False)
    return results_df


In [ ]:
all_results = 
print(all_results.head())

In [ ]:


# def get_alternative(product_name, df_A1, similarity_matrix, threshold=0.90):
#     product_name = product_name.strip().lower()
#     df_A1["inv_scan"] = df_A1["inv_scancode"].str.strip().str.lower()

#     if product_name not in df_A1["inv_scancode"].values:
#         return pd.DataFrame(columns=["inv_pk", "inv_scancode", "original_product", "similar_product", "similarity_score", "similar_inv_pk", "similar_brd_name", "similar_dpt_name"])

#     idx = df_A1[df_A1["inv_scancode"] == product_name].index[0]

#     # Ensure index is valid
#     if idx >= similarity_matrix.shape[0]:
#         return pd.DataFrame(columns=["inv_pk", "inv_scancode", "original_product", "similar_product", "similarity_score", "similar_inv_pk", "similar_brd_name", "similar_dpt_name"])

#     # Compute cosine similarity
#     similarity_matrix = cosine_similarity([embeddings[idx]], embeddings)[0]

#     # Get original product details
#     original_inv_pk = df_A1.iloc[idx]["inv_pk"]  # Primary key for original product
#     original_brd_name = df_A1.iloc[idx]["brd_name"] if "brd_name" in df_A1.columns else None

#     # Get similar products
#     similar_products = [
#         (df_A1.iloc[i]["inv_scancode"],  
#             df_A1.iloc[i]["inv_pk"],  
#             bert_similarity_matrix[i],  
#             df_A1.iloc[i]["brd_name"] if "brd_name" in df_A1.columns else None,  
#             df_A1.iloc[i]["dpt_name"] if "dpt_name" in df_A1.columns else None)
#         for i in range(len(df_A1)) if i != idx and bert_similarity_matrix[i] >= threshold
#     ]

#     # Sort by similarity score and take top N
#     similar_products = sorted(similar_products, key=lambda x: x[1], reverse=True)[:top_n]


#     # Convert to DataFrame
#     similarity_df = pd.DataFrame(similar_products, columns=["similar_product", "similar_inv_pk", "similarity_score", "similar_brd_name", "similar_dpt_name"])

#      # Add original product details
#     similarity_df["original_product"] = product_name
#     similarity_df["inv_pk"] = original_inv_pk  # Assigning without inserting
#     similarity_df["brd_name"] = original_brd_name  # Assigning without inserting


#     return similarity_df

In [ ]:
df_A1